In [0]:
# 1. Clear everything
catalog = "capstone_101"
checkpoint_path = "/Volumes/capstone_101/bronze/raw_files/_checkpoints/bronze_ingest"

dbutils.fs.rm(checkpoint_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.raw_sales")

# 2. Re-ingest with explicit STRING type for CustomerID
from pyspark.sql.functions import current_timestamp, col

bronze_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema") 
    .option("cloudFiles.inferColumnTypes", "false") # Force STRING for all columns
    .option("header", "true")
    .load("/Volumes/capstone_101/bronze/raw_files/")
    .select("*", 
            col("_metadata.file_path").alias("_input_file_name"), 
            current_timestamp().alias("_ingest_timestamp"))
)

query = (bronze_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true") 
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.raw_sales"))

query.awaitTermination()
print("Bronze table recreated successfully with STRING schema.")